In [14]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install scipy

   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------------ --------------------------- 11.0/36.5 MB 63.1 MB/s eta 0:00:01
   --------------------------- ------------ 25.4/36.5 MB 68.0 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 62.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import h5py
import numpy as np
import os
import torch

# -----------------------------
# Paths
# -----------------------------

base_dir = r"C:/Users/Micah/utah-neuro/MATLAB"

kdf_path = os.path.join(base_dir, "kdf(andother).h5")
ns5_training_path = os.path.join(base_dir, "ns5_training.h5")
output_path = os.path.join(base_dir, "Aligned_Train_Data.pt")

# -----------------------------
# Load KDF training data
# -----------------------------

with h5py.File(kdf_path, "r") as kdf:
    train_niptime = kdf["trainNIPtime"][:].flatten()
    train_kinematics = kdf["trainKin"][:]

# Keep only first 7 kinematic values
train_kinematics = train_kinematics[:, :7]

# Sort by trainNIPtime
sort_idx = np.argsort(train_niptime)
train_niptime_sorted = train_niptime[sort_idx]
train_kinematics_sorted = train_kinematics[sort_idx]

# -----------------------------
# Load full NS5 training data
# -----------------------------

with h5py.File(ns5_training_path, "r") as f:
    print("NS5 datasets:", list(f.keys()))
    ns5_training = f["data"][:]

# Make sure shape is samples x channels
if ns5_training.shape[0] == 32:
    ns5_training = ns5_training.T

print("ns5_training shape:", ns5_training.shape)

num_ns5_samples = ns5_training.shape[0]

# -----------------------------
# Convert trainNIPtime to row index inside ns5_training
# -----------------------------

train_niptime_start = train_niptime_sorted[0]

kdf_indices_0indexed = (
    train_niptime_sorted - train_niptime_start
).astype(np.int64)

if kdf_indices_0indexed.min() < 0:
    raise ValueError("Some KDF indices are below 0.")

if kdf_indices_0indexed.max() >= num_ns5_samples:
    raise ValueError(
        f"Some KDF indices exceed ns5_training length. "
        f"Max requested index: {kdf_indices_0indexed.max()}, "
        f"NS5 samples available: {num_ns5_samples}"
    )

# -----------------------------
# Fill every NS5 sample with preceding KDF trainKin
# -----------------------------

all_ns5_indices = np.arange(num_ns5_samples)

# For each NS5 sample, find most recent KDF index at or before it
preceding_kdf_pos = np.searchsorted(
    kdf_indices_0indexed,
    all_ns5_indices,
    side="right"
) - 1

# Remove NS5 samples before the first KDF timestamp, if any
valid_mask = preceding_kdf_pos >= 0

all_ns5_indices = all_ns5_indices[valid_mask]
preceding_kdf_pos = preceding_kdf_pos[valid_mask]

filled_train_kinematics = train_kinematics_sorted[preceding_kdf_pos]
filled_train_niptime = train_niptime_start + all_ns5_indices

filled_ns5_vectors = ns5_training[all_ns5_indices, :]

# Store 1-indexed row numbers inside ns5_training
ns5_samples_1indexed = all_ns5_indices + 1

# -----------------------------
# Convert matched data to PyTorch tensors
# -----------------------------

aligned_tensor = {
    "ns5_sample": torch.tensor(ns5_samples_1indexed, dtype=torch.long),
    "trainNIPtime": torch.tensor(filled_train_niptime, dtype=torch.long),
    "ns5_vector": torch.tensor(filled_ns5_vectors, dtype=torch.float32),
    "trainKin": torch.tensor(filled_train_kinematics, dtype=torch.float32),
}

# -----------------------------
# One-hot encode trainKin
# -----------------------------

def argmax_one_hot_keep_zeros(x):
    max_indices = torch.argmax(x, dim=1)
    one_hot = torch.zeros_like(x)

    nonzero_rows = x.sum(dim=1) != 0
    rows = torch.arange(x.shape[0], device=x.device)

    one_hot[rows[nonzero_rows], max_indices[nonzero_rows]] = 1
    return one_hot

aligned_tensor["trainKin"] = argmax_one_hot_keep_zeros(aligned_tensor["trainKin"])

# -----------------------------
# Save tensor file
# -----------------------------

torch.save(aligned_tensor, output_path)

# -----------------------------
# Print checks
# -----------------------------

print("Number of full NS5 samples used:", len(aligned_tensor["trainNIPtime"]))
print("ns5_vector shape:", aligned_tensor["ns5_vector"].shape)
print("trainKin shape:", aligned_tensor["trainKin"].shape)

print("\nSaved tensor file to:")
print(output_path)

NS5 datasets: ['data']
ns5_training shape: (5851861, 32)
Number of full NS5 samples used: 5851861
ns5_vector shape: torch.Size([5851861, 32])
trainKin shape: torch.Size([5851861, 7])

First 10 ns5_sample values:
tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

First 10 trainNIPtime values:
tensor([35943660, 35943661, 35943662, 35943663, 35943664, 35943665, 35943666,
        35943667, 35943668, 35943669])

Saved tensor file to:
C:/Users/Micah/utah-neuro/MATLAB\Aligned_Train_Data.pt
